# TAMER OCR v2.2 — Multi-Account Orchestrator

This notebook contains **no logic**. It is a pure orchestration layer designed to allow seamless session-hopping across different Kaggle and Google Colab accounts.

**How it works:**
1. **Code Sync:** It always forces a `git pull` to ensure you have the latest architectural fixes from GitHub.
2. **Data Smart-Resume:** If it finds local cached data (e.g., interrupted preprocessing), it picks up exactly where it left off.
3. **Account Hopping:** If run on a brand new account, it downloads the already-processed dataset and the latest model checkpoint directly from Hugging Face.
4. **Execution:** It hands control to `Trainer.run()`.

## 1. Clean Notebook Space & Auth

In [ ]:
import os
import getpass

# Remove old floating notebooks (but KEEP data and checkpoints)
!rm -f *.ipynb

# Detect Environment
IS_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
os.chdir(WORK_DIR)

print(f"🌍 Environment: {'Kaggle' if IS_KAGGLE else 'Colab'}")

print("\n🔑 AUTHENTICATION")
# Hugging Face Token
hf_token = os.getenv('HF_TOKEN', '')
if not hf_token:
    hf_token = getpass.getpass('Enter HuggingFace Token (WRITE access needed): ')
os.environ['HF_TOKEN'] = hf_token

# Kaggle Credentials
kaggle_username = os.getenv('KAGGLE_USERNAME', '')
if not kaggle_username:
    kaggle_username = input('Enter Kaggle Username: ').strip()
    
kaggle_key = os.getenv('KAGGLE_KEY', '')
if not kaggle_key:
    kaggle_key = getpass.getpass('Enter Kaggle API Key: ')
    
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

# Configure Kaggle CLI
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print("✅ Authentication complete.")

## 2. Dependencies & Codebase Sync
This cell checks if the repo exists. If it does, it forces an update to get your new architecture. If it doesn't (new account), it clones it.

In [ ]:
print("📦 Installing dependencies...")
!pip install -q timm albumentations datasets huggingface_hub requests tqdm psutil opencv-python-headless

print("\n📥 Syncing Codebase...")
REPO_URL = "https://github.com/Vtheonly/AI_PROJECT_TAMER_Complete"
REPO_DIR = "AI_PROJECT_TAMER_Complete"

if os.path.exists(REPO_DIR):
    print("Repo found. Pulling latest architecture fixes...")
    %cd {REPO_DIR}
    !git reset --hard origin/main
    !git pull
    %cd ..
else:
    print("Cloning repo for the first time...")
    !git clone {REPO_URL}
print("✅ Codebase synchronized.")

## 3. Configuration & System Path

In [ ]:
import sys

REPO_ROOT = os.path.join(WORK_DIR, REPO_DIR)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from tamer_ocr.config import Config
from huggingface_hub import HfApi

config = Config()
config.data_dir = os.path.join(WORK_DIR, 'data')
config.output_dir = os.path.join(WORK_DIR, 'outputs')
config.checkpoint_dir = os.path.join(WORK_DIR, 'checkpoints')
config.log_dir = os.path.join(WORK_DIR, 'logs')

for d in [config.data_dir, config.output_dir, config.checkpoint_dir, config.log_dir]:
    os.makedirs(d, exist_ok=True)

hf_api = HfApi(token=os.environ['HF_TOKEN'])
hf_username = hf_api.whoami()['name']
config.hf_token = os.environ['HF_TOKEN']
config.hf_repo_id = f"{hf_username}/tamer-ocr-model"
config.hf_dataset_repo_id = f"{hf_username}/tamer-preprocessed"

# Fast Training Setup
config.batch_size = 8
config.accumulation_steps = 4
config.num_epochs = 150
config.encoder_lr = 1e-5
config.decoder_lr = 1e-4
config.checkpoint_every_epochs = 3
config.keep_last_n_checkpoints = 3

print("✅ Pipeline configured.")

## 4. Retrieve Latest Checkpoint (If Switching Accounts)
If you start this on a new account, it will automatically pull your latest model weights from Hugging Face so you don't start from Epoch 0.

In [ ]:
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import EntryNotFoundError

try:
    print("🔍 Checking Hugging Face for existing checkpoints...")
    # Try to grab the 'best.pt' first
    downloaded_path = hf_hub_download(
        repo_id=config.hf_repo_id, 
        filename="best.pt", 
        token=config.hf_token,
        repo_type="model",
        local_dir=config.checkpoint_dir
    )
    print(f"✅ Found remote checkpoint! Downloaded to {downloaded_path}")
except EntryNotFoundError:
    print("ℹ️ No existing checkpoint found on Hugging Face. Starting fresh or using local.")
except Exception as e:
    print(f"ℹ️ Could not check HF (Repo might not exist yet). Message: {e}")

## 5. Execute Training Pipeline

In [ ]:
from tamer_ocr.core.trainer import Trainer

print("🚀 Launching TAMER Orchestrator...")
trainer = Trainer(config)
trainer.run()  # Will detect cached data, skip completed preprocessing, and train.

## 6. Manual Hub Push (Failsafe)
The trainer auto-pushes, but this ensures your best model is absolutely synced before you close the window.

In [ ]:
from tamer_ocr.utils.checkpoint import push_checkpoint_to_hf

best_model_path = os.path.join(config.checkpoint_dir, 'best.pt')
if os.path.exists(best_model_path):
    print(f"📤 Syncing best model to {config.hf_repo_id}...")
    push_checkpoint_to_hf(best_model_path, config, epoch=0, is_best=True)
    print("✅ Sync complete.")